In [1]:
import pandas as pd
import kagglehub
import ast
import numpy as np

path = kagglehub.dataset_download("grouplens/movielens-latest-full")

print("Path to dataset files:", path)

Path to dataset files: /Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1


In [2]:
import glob
file_list = glob.glob("/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/*")

In [3]:
file_list

['/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/links.csv',
 '/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/tags.csv',
 '/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/genome-tags.csv',
 '/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/README.md',
 '/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/ratings.csv',
 '/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/genome-scores.csv',
 '/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/movies.csv']

In [4]:
pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/genome-scores.csv')

,movieId,tagId,relevance
0,1,1,0.02900
1,1,2,0.02375
2,1,3,0.05425
3,1,4,0.06875
4,1,5,0.16000
...,...,...,...
14862523,187595,1124,0.10700
14862524,187595,1125,0.05850
14862525,187595,1126,0.03800
14862526,187595,1127,0.10225


In [5]:
path = kagglehub.dataset_download("rounakbanik/the-movies-dataset")

print("Path to dataset files:", path)

Path to dataset files: /Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7


In [6]:
glob.glob(f"{path}/*")

['/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/links_small.csv',
 '/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/links.csv',
 '/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/credits.csv',
 '/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/movies_metadata.csv',
 '/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/ratings.csv',
 '/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/ratings_small.csv',
 '/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/keywords.csv']

In [24]:
rating_stats = ratings_ml.groupby('movieId')['rating'].agg(['mean', 'min', 'max', 'count']).reset_index()
rating_stats.columns = ['movieId', 'vote_average', 'vote_min', 'vote_max', 'vote_count']

In [25]:
rating_stats

,movieId,vote_average,vote_min,vote_max,vote_count
0,1,3.886649,0.5,5.0,68469
1,2,3.246583,0.5,5.0,27143
2,3,3.173981,0.5,5.0,15585
3,4,2.874540,0.5,5.0,2989
4,5,3.077291,0.5,5.0,15474
...,...,...,...,...,...
53884,193876,3.000000,3.0,3.0,1
53885,193878,2.000000,2.0,2.0,1
53886,193880,2.000000,2.0,2.0,1
53887,193882,2.000000,2.0,2.0,1


In [ ]:
# MovieLens
ratings_ml = pd.read_csv( '/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/ratings.csv')         # 25M ratings
movies_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/movies.csv')           # titles and genres
links_ml = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/links.csv')             # maps movieId to imdbId, tmdbId
tags_ml = pd.read_csv( '/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/tags.csv',)
tags_agg = tags_ml.groupby('movieId')['tag'].apply(lambda x: list(set(x))).reset_index()

# TMDB metadata
metadata_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/movies_metadata.csv', low_memory=False)
credits_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/credits.csv')
keywords_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/keywords.csv')
links_tmdb = pd.read_csv('/Users/aniket/.cache/kagglehub/datasets/rounakbanik/the-movies-dataset/versions/7/links.csv')

In [8]:
links_ml = links_ml[links_ml['tmdbId'].notnull()]
links_ml['tmdbId'] = links_ml['tmdbId'].astype(int)

metadata_tmdb = metadata_tmdb[pd.to_numeric(metadata_tmdb['id'], errors='coerce').notnull()]
metadata_tmdb['id'] = metadata_tmdb['id'].astype(int)

metadata_tmdb['genres'] = metadata_tmdb['genres'].fillna('[]').apply(ast.literal_eval)
metadata_tmdb['main_genre'] = metadata_tmdb['genres'].apply(lambda x: x[0]['name'] if isinstance(x, list) and x else None)

movies_ml_links = pd.merge(movies_ml, links_ml, on='movieId')
metadata_tmdb = metadata_tmdb.rename(columns={'id': 'tmdbId'})
movies_full = pd.merge(movies_ml_links, metadata_tmdb, on='tmdbId', how='inner')

credits_tmdb['cast'] = credits_tmdb['cast'].apply(ast.literal_eval)
credits_tmdb['crew'] = credits_tmdb['crew'].apply(ast.literal_eval)

def get_director(crew):
    for person in crew:
        if person.get('job') == 'Director':
            return person.get('name')
    return None

def get_lead_actor(cast):
    return cast[0]['name'] if isinstance(cast, list) and cast else None

credits_tmdb['tmdbId'] = credits_tmdb['id']
credits_tmdb['director'] = credits_tmdb['crew'].apply(get_director)
credits_tmdb['lead_actor'] = credits_tmdb['cast'].apply(get_lead_actor)

movies_full = pd.merge(movies_full, credits_tmdb[['tmdbId', 'director', 'lead_actor']], on='tmdbId', how='left')

keywords_tmdb['keywords'] = keywords_tmdb['keywords'].fillna('[]').apply(ast.literal_eval)
keywords_tmdb['keywords'] = keywords_tmdb['keywords'].apply(lambda x: [d['name'] for d in x if isinstance(d, dict)])
keywords_tmdb['tmdbId'] = keywords_tmdb['id']

movies_full = pd.merge(movies_full, keywords_tmdb[['tmdbId', 'keywords']], on='tmdbId', how='left')

movies_full = movies_full.rename(columns={
    'title_x': 'title_ml',
    'title_y': 'title_tmdb',
    'genres_x': 'genres_ml',
    'genres_y': 'genres_tmdb'
})


In [9]:
movies_full.columns

Index(['movieId', 'title_ml', 'genres_ml', 'imdbId', 'tmdbId', 'adult',
       'belongs_to_collection', 'budget', 'genres_tmdb', 'homepage', 'imdb_id',
       'original_language', 'original_title', 'overview', 'popularity',
       'poster_path', 'production_companies', 'production_countries',
       'release_date', 'revenue', 'runtime', 'spoken_languages', 'status',
       'tagline', 'title_tmdb', 'video', 'vote_average', 'vote_count',
       'main_genre', 'director', 'lead_actor', 'keywords'],
      dtype='object')

In [10]:
movies_full.info(memory_usage="deep")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46898 entries, 0 to 46897
Data columns (total 32 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   movieId                46898 non-null  int64  
 1   title_ml               46898 non-null  object 
 2   genres_ml              46898 non-null  object 
 3   imdbId                 46898 non-null  int64  
 4   tmdbId                 46898 non-null  int64  
 5   adult                  46898 non-null  object 
 6   belongs_to_collection  4600 non-null   object 
 7   budget                 46898 non-null  object 
 8   genres_tmdb            46898 non-null  object 
 9   homepage               8049 non-null   object 
 10  imdb_id                46881 non-null  object 
 11  original_language      46887 non-null  object 
 12  original_title         46898 non-null  object 
 13  overview               45906 non-null  object 
 14  popularity             46894 non-null  object 
 15  po

In [11]:
# classifier - ecplaim n compare after removing features

In [12]:
# daniel hess

In [13]:
movies_full = pd.merge(movies_full, tags_agg, on='movieId', how='left')

In [14]:
# genome_tags = pd.read_csv( '/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/genome-tags.csv')
# genome_scores = pd.read_csv( '/Users/aniket/.cache/kagglehub/datasets/grouplens/movielens-latest-full/versions/1/genome-scores.csv')

# # Pivot for each movieId: each tag becomes a column with its relevance score
# genome_pivot = genome_scores.pivot(index='movieId', columns='tagId', values='relevance')
# genome_pivot.columns = ['genome_' + str(tag_id) for tag_id in genome_pivot.columns]

# movies_full = pd.merge(movies_full, genome_pivot, left_on='movieId', right_index=True, how='left')


In [ ]:
movies_full['release_year'] = pd.to_datetime(movies_full['release_date'], errors='coerce').dt.year

bin_edges = list(range(0, 301, 30)) + [np.inf]


labels = [f'{bin_edges[i]}–{bin_edges[i+1]}min' if bin_edges[i+1] != np.inf else f'{bin_edges[i]}min+'
          for i in range(len(bin_edges) - 1)]

# Apply to your DataFrame
movies_full['runtime_bin'] = pd.cut(movies_full['runtime'], bins=bin_edges, labels=labels)

movies_full

,movieId,title_ml,genres_ml,imdbId,tmdbId,adult,belongs_to_collection,budget,genres_tmdb,homepage,...,video,vote_average,vote_count,main_genre,director,lead_actor,keywords,tag,release_year,runtime_bin
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,114709,862,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,...,False,7.7,5415.0,Animation,John Lasseter,Tom Hanks,"[jealousy, toy, boy, friendship, friends, riva...","[family, CGI classic, humorous, light, CG anim...",1995.0,60–90min
1,2,Jumanji (1995),Adventure|Children|Fantasy,113497,8844,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,...,False,6.9,2413.0,Adventure,Joe Johnston,Robin Williams,"[board game, disappearance, based on children'...","[rainy day watchlist, animals, family, Chris V...",1995.0,90–120min
2,3,Grumpier Old Men (1995),Comedy|Romance,113228,15602,False,"{'id': 119050, 'name': 'Grumpy Old Men Collect...",0,"[{'id': 10749, 'name': 'Romance'}, {'id': 35, ...",NaN,...,False,6.5,92.0,Romance,Howard Deutch,Walter Matthau,"[fishing, best friend, duringcreditsstinger, o...","[best friend, good soundtrack, CLV, old man, s...",1995.0,90–120min
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,114885,31357,False,NaN,16000000,"[{'id': 35, 'name': 'Comedy'}, {'id': 18, 'nam...",NaN,...,False,6.1,34.0,Comedy,Forest Whitaker,Whitney Houston,"[based on novel, interracial relationship, sin...","[chick flick, characters, single mother, inter...",1995.0,120–150min
4,5,Father of the Bride Part II (1995),Comedy,113041,11862,False,"{'id': 96871, 'name': 'Father of the Bride Col...",0,"[{'id': 35, 'name': 'Comedy'}]",NaN,...,False,5.7,173.0,Comedy,Charles Shyer,Steve Martin,"[baby, midlife crisis, confidence, aging, daug...","[Comedy, worst movies ever, childhood classics...",1995.0,90–120min
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46893,181393,Venice (2010),Drama|Romance,1684935,79782,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 10749, 'n...",NaN,...,False,7.5,4.0,Drama,Jan Jakub Kolski,Marcin Walewski,[],NaN,2010.0,90–120min
46894,181751,Lagaan: Once Upon a Time in India (2001),Adventure|Drama|Romance,169102,19666,False,NaN,5200000,"[{'id': 12, 'name': 'Adventure'}, {'id': 18, '...",http://www.lagaan.com,...,False,7.2,125.0,Adventure,Ashutosh Gowariker,Aamir Khan,"[sport, british, bollywood, arrogance, based o...","[arrogance, 19th century, british, sport, base...",2001.0,210–240min
46895,183123,Taboo (2002),Drama|Horror|Mystery|Thriller,288243,97206,False,NaN,0,"[{'id': 18, 'name': 'Drama'}, {'id': 27, 'name...",NaN,...,False,2.9,9.0,Drama,Max Makowski,Nick Stahl,[],NaN,2002.0,60–90min
46896,187127,David Lynch: The Art Life (2017),Documentary,1691152,413765,False,NaN,0,"[{'id': 99, 'name': 'Documentary'}]",NaN,...,False,7.2,32.0,Documentary,Olivia Neergaard-Holm,David Lynch,[],[Criterion],2017.0,60–90min


In [16]:
movies_full.columns

Index(['movieId', 'title_ml', 'genres_ml', 'imdbId', 'tmdbId', 'adult',
       'belongs_to_collection', 'budget', 'genres_tmdb', 'homepage', 'imdb_id',
       'original_language', 'original_title', 'overview', 'popularity',
       'poster_path', 'production_companies', 'production_countries',
       'release_date', 'revenue', 'runtime', 'spoken_languages', 'status',
       'tagline', 'title_tmdb', 'video', 'vote_average', 'vote_count',
       'main_genre', 'director', 'lead_actor', 'keywords', 'tag',
       'release_year', 'runtime_bin'],
      dtype='object')

In [17]:
stop code

SyntaxError: invalid syntax (2910947498.py, line 1)

In [ ]:
# Fix numeric fields
metadata_tmdb = metadata_tmdb[pd.to_numeric(metadata_tmdb['id'], errors='coerce').notnull()]
metadata_tmdb['id'] = metadata_tmdb['id'].astype(int)

# Parse genres
import ast
metadata_tmdb['genres'] = metadata_tmdb['genres'].fillna('[]').apply(ast.literal_eval)
metadata_tmdb['main_genre'] = metadata_tmdb['genres'].apply(lambda x: x[0]['name'] if x else None)


In [ ]:
links_ml = links_ml[pd.notnull(links_ml['tmdbId'])]   # remove rows without TMDB ID
links_ml['tmdbId'] = links_ml['tmdbId'].astype(int)


In [ ]:
# Step 1: Merge MovieLens movies with TMDB IDs
movies_ml_links = pd.merge(movies_ml, links_ml, on='movieId')

# Step 2: Merge that with TMDB metadata using tmdbId = id
movies_full = pd.merge(movies_ml_links, metadata_tmdb, left_on='tmdbId', right_on='id')


In [ ]:
# Parse JSON columns
credits_tmdb['cast'] = credits_tmdb['cast'].apply(ast.literal_eval)
credits_tmdb['crew'] = credits_tmdb['crew'].apply(ast.literal_eval)

# Extract director and lead actor
def get_director(crew):
    for person in crew:
        if person['job'] == 'Director':
            return person['name']
    return None

def get_lead_actor(cast):
    return cast[0]['name'] if cast else None

credits_tmdb['director'] = credits_tmdb['crew'].apply(get_director)
credits_tmdb['lead_actor'] = credits_tmdb['cast'].apply(get_lead_actor)

# Parse keywords
keywords_tmdb['keywords'] = keywords_tmdb['keywords'].fillna('[]').apply(ast.literal_eval)
keywords_tmdb['keywords'] = keywords_tmdb['keywords'].apply(lambda x: [d['name'] for d in x])

# Merge into full metadata
movies_full = pd.merge(movies_full, credits_tmdb[['id', 'director', 'lead_actor']], left_on='tmdbId', right_on='id', how='left')
movies_full = pd.merge(movies_full, keywords_tmdb[['id', 'keywords']], left_on='tmdbId', right_on='id', how='left')


In [ ]:
movies_full.columns

In [ ]:
movies_full["id_y"]

In [ ]:
len(movies_full["tmdbId"].unique())

In [ ]:
len(movies_full["imdbId"].unique())

In [ ]:
len(movies_full["movieId"].unique())

In [ ]:
movies_full.columns

In [ ]:
keywords_tmdb.columns

In [ ]:
credits_tmdb.columns